# Phase II: Original LCS Baseline



## 1. Environment and Reproducibility



## 2. Load Original Dataset




In [2]:
import pandas as pd

df = pd.read_csv("../data/telecom_customer_churn.csv")

print("Dataset loaded successfully")
print("Shape:", df.shape)

df.head()

Dataset loaded successfully
Shape: (7043, 38)


,Customer ID,Gender,Age,Married,Number of Dependents,City,Zip Code,Latitude,Longitude,Number of Referrals,...,Payment Method,Monthly Charge,Total Charges,Total Refunds,Total Extra Data Charges,Total Long Distance Charges,Total Revenue,Customer Status,Churn Category,Churn Reason
0,0002-ORFBO,Female,37,Yes,0,Frazier Park,93225,34.827662,-118.999073,2,...,Credit Card,65.6,593.30,0.00,0,381.51,974.81,Stayed,NaN,NaN
1,0003-MKNFE,Male,46,No,0,Glendale,91206,34.162515,-118.203869,0,...,Credit Card,-4.0,542.40,38.33,10,96.21,610.28,Stayed,NaN,NaN
2,0004-TLHLJ,Male,50,No,0,Costa Mesa,92627,33.645672,-117.922613,0,...,Bank Withdrawal,73.9,280.85,0.00,0,134.60,415.45,Churned,Competitor,Competitor had better devices
3,0011-IGKFF,Male,78,Yes,0,Martinez,94553,38.014457,-122.115432,1,...,Bank Withdrawal,98.0,1237.85,0.00,0,361.66,1599.51,Churned,Dissatisfaction,Product dissatisfaction
4,0013-EXCHZ,Female,75,Yes,0,Camarillo,93010,34.227846,-119.079903,3,...,Credit Card,83.9,267.40,0.00,0,22.14,289.54,Churned,Dissatisfaction,Network reliability


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 38 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   Customer ID                        7043 non-null   str    
 1   Gender                             7043 non-null   str    
 2   Age                                7043 non-null   int64  
 3   Married                            7043 non-null   str    
 4   Number of Dependents               7043 non-null   int64  
 5   City                               7043 non-null   str    
 6   Zip Code                           7043 non-null   int64  
 7   Latitude                           7043 non-null   float64
 8   Longitude                          7043 non-null   float64
 9   Number of Referrals                7043 non-null   int64  
 10  Tenure in Months                   7043 non-null   int64  
 11  Offer                              3166 non-null   str    
 12  Pho

In [5]:
for col in df.columns:
    print(col)

    

Customer ID
Gender
Age
Married
Number of Dependents
City
Zip Code
Latitude
Longitude
Number of Referrals
Tenure in Months
Offer
Phone Service
Avg Monthly Long Distance Charges
Multiple Lines
Internet Service
Internet Type
Avg Monthly GB Download
Online Security
Online Backup
Device Protection Plan
Premium Tech Support
Streaming TV
Streaming Movies
Streaming Music
Unlimited Data
Contract
Paperless Billing
Payment Method
Monthly Charge
Total Charges
Total Refunds
Total Extra Data Charges
Total Long Distance Charges
Total Revenue
Customer Status
Churn Category
Churn Reason


In [6]:
for col in df.columns:
    if "churn" in col.lower():
        print(col)

Churn Category
Churn Reason


In [7]:
df["Customer Status"].value_counts(dropna=False)

Customer Status
Stayed     4720
Churned    1869
Joined      454
Name: count, dtype: int64

## 3. Define Target and Leakage Variables



In [8]:
df["Churn Target"] = (
    df["Customer Status"] == "Churned"
).astype(int)

print("Binary target distribution:")
print(df["Churn Target"].value_counts())

print("\nBinary target percentages:")
print(
    df["Churn Target"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

Binary target distribution:
Churn Target
0    5174
1    1869
Name: count, dtype: int64

Binary target percentages:
Churn Target
0    73.46
1    26.54
Name: proportion, dtype: float64


In [9]:
leakage_columns = [
    "Customer Status",
    "Churn Category",
    "Churn Reason"
]

existing_leakage_columns = [
    col for col in leakage_columns
    if col in df.columns
]

print("Target-leakage columns:")
print(existing_leakage_columns)

Target-leakage columns:
['Customer Status', 'Churn Category', 'Churn Reason']


In [10]:
columns_to_exclude = [
    "Customer ID",
    "Customer Status",
    "Churn Category",
    "Churn Reason",
    "Churn Target"
]

columns_to_exclude = [
    col for col in columns_to_exclude
    if col in df.columns
]

X_raw = df.drop(columns=columns_to_exclude)
y = df["Churn Target"]

print("Raw feature shape:", X_raw.shape)
print("Target shape:", y.shape)
print("Excluded columns:", columns_to_exclude)

Raw feature shape: (7043, 34)
Target shape: (7043,)
Excluded columns: ['Customer ID', 'Customer Status', 'Churn Category', 'Churn Reason', 'Churn Target']


### Target Definition

Customer Status contains three categories: Stayed, Churned, and Joined. A binary target was created for the churn-classification task. Churned customers were assigned a value of 1, while Stayed and Joined customers were assigned a value of 0.

Customer Status, Churn Category, and Churn Reason were excluded from the predictor variables because they contain direct or post-outcome information about customer churn and could introduce target leakage. Customer ID was also excluded because it is an identifier rather than a predictive feature.

## 4. Minimal LCS Compatibility Processing



## 5. Stratified Train-Test Split



In [11]:
from sklearn.model_selection import train_test_split

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training features:", X_train_raw.shape)
print("Testing features:", X_test_raw.shape)

print("\nTraining target distribution:")
print(y_train.value_counts())

print("\nTesting target distribution:")
print(y_test.value_counts())

Training features: (5634, 34)
Testing features: (1409, 34)

Training target distribution:
Churn Target
0    4139
1    1495
Name: count, dtype: int64

Testing target distribution:
Churn Target
0    1035
1     374
Name: count, dtype: int64


In [12]:
print("Training percentages:")
print((y_train.value_counts(normalize=True) * 100).round(2))

print("\nTesting percentages:")
print((y_test.value_counts(normalize=True) * 100).round(2))

Training percentages:
Churn Target
0    73.46
1    26.54
Name: proportion, dtype: float64

Testing percentages:
Churn Target
0    73.46
1    26.54
Name: proportion, dtype: float64


### Train-Test Strategy

An 80/20 train-test split was used for the baseline experiments. Stratification was applied to preserve the class distribution of churned and non-churned customers in both training and testing datasets. A fixed random seed (42) was used to ensure reproducibility.

## 6. Original LCS Configuration



## 7. Baseline Training



## 8. Baseline Evaluation



## 9. Export Results

In [1]:
import sys
import platform
import pandas as pd
import numpy as np
import sklearn
import scipy

print("Python:", sys.version)
print("Platform:", platform.platform())
print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("scikit-learn:", sklearn.__version__)
print("scipy:", scipy.__version__)

Python: 3.14.7 (tags/v3.14.7:823f032, Aug  5 2026, 10:51:32) [MSC v.1944 64 bit (AMD64)]
Platform: Windows-11-10.0.22631-SP0
pandas: 3.0.5
numpy: 2.5.2
scikit-learn: 1.9.1
scipy: 1.18.1
